# Phase 6 · Cancer care: the radiotherapy register and one radiotherapy centre

> **An illustrative model using public data and stated assumptions. Not investment advice.**

The same two steps as kidney care, with the same code:
1. **Assess:** a hand-built register of radiotherapy capacity, one row per fact per source.
2. **What one centre could do:** patients treated, capacity used and break-even for one illustrative
   radiotherapy centre.

Terms:
- **Radiotherapy**: cancer treatment with radiation, from a **linac** (linear accelerator) or a cobalt-60 unit.
- **Course** and **fraction**: a patient has a course of many daily treatments, called fractions, over a few weeks.
- **Oncologist**: a cancer specialist (radiation or clinical oncologist).

Every number below is printed by code.

In [1]:
import sys, math
from pathlib import Path
import pandas as pd
import plotly.io as pio

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "pipeline"), str(ROOT / "src")]
import centre_model as M, charts, story, validate

pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_rows", 60, "display.width", 180, "display.max_colwidth", 100)
P = ROOT / "data" / "processed"
reg = pd.read_csv(P / "register_cancer_rows.csv", dtype=str, keep_default_na=False)
by_state = pd.read_csv(P / "register_cancer_by_state.csv")
nat = pd.read_csv(P / "register_cancer_national.csv", dtype=str, keep_default_na=False)
panel = pd.read_csv(P / "state_panel.csv")

## 1. The register at a glance
Rules as for kidney: conflicting counts stay as separate rows; `gap_type` comes only from what the source says
(otherwise `unknown`); `status` "installed" is not "working".

In [2]:
print(f"{len(reg)} cancer rows from {reg.source_url.nunique()} source pages; "
      f"{int((by_state.evidence == 'found').sum())} of 37 states have any evidence.")
for col in ["source_type", "item", "status", "gap_type"]:
    print(f"  {col:12}", reg[col].value_counts().to_dict())
print(f"\nRows with gap_type 'unknown': {int((reg.gap_type == 'unknown').sum())} of {len(reg)}")

100 cancer rows from 41 source pages; 16 of 37 states have any evidence.
  source_type  {'peer-reviewed': 53, 'news': 35, 'government': 11, 'professional body': 1}
  item         {'other': 38, 'linac': 34, 'oncologist': 16, 'cobalt unit': 12}
  status       {'unknown': 49, 'working': 35, 'not working': 16}
  gap_type     {'unknown': 69, 'systems': 11, 'workforce': 11, 'equipment': 9}

Rows with gap_type 'unknown': 69 of 100


In [3]:
cols = ["state", "n_rows", "public_unit_working", "machines_working_low", "machines_working_high",
        "machines_not_working_high", "specialists_high", "gap_equipment", "gap_workforce", "gap_systems", "gap_unknown"]
by_state[by_state.evidence == "found"][cols]

,state,n_rows,public_unit_working,machines_working_low,machines_working_high,machines_not_working_high,specialists_high,gap_equipment,gap_workforce,gap_systems,gap_unknown
7,Borno,6,yes,2.0,2.0,NaN,1.0,1.0,0.0,0.0,5.0
8,Cross River,3,unknown,NaN,NaN,NaN,NaN,0.0,0.0,0.0,3.0
10,Ebonyi,1,unknown,NaN,NaN,NaN,NaN,0.0,0.0,0.0,1.0
11,Edo,4,yes,NaN,NaN,1.0,NaN,0.0,0.0,1.0,3.0
13,Enugu,5,yes,NaN,NaN,NaN,3.0,0.0,0.0,2.0,3.0
14,Federal Capital Territory,6,yes,2.0,2.0,1.0,3.0,0.0,1.0,2.0,3.0
15,Gombe,4,yes,NaN,NaN,NaN,3.0,0.0,0.0,1.0,3.0
16,Imo,1,unknown,NaN,NaN,NaN,NaN,0.0,0.0,0.0,1.0
18,Kaduna,5,reported not working,NaN,NaN,1.0,5.0,1.0,1.0,0.0,3.0
19,Kano,6,reported not working,NaN,NaN,2.0,NaN,2.0,1.0,1.0,2.0


In [4]:
lo, hi, n = story.machines_range(by_state)
print(f"Working machines named in the register: {lo:.0f}–{hi:.0f}, in {n} states with a count.")
print(story.public_unit_sentence(by_state, "radiotherapy"))

Working machines named in the register: 10–10, in 5 states with a count.
7 of 37 states have a public radiotherapy unit reported working in the register; in 2 the only reports say it isn't working, and for 28 the register has no evidence either way.


## 2. The national counts don't agree
Grouped by rule (not hand-picked): working machines of any type, working linacs, oncologists, machines needed.

In [5]:
for k, g in charts.cancer_national_groups(nat).items():
    vals = ", ".join(f"{v:,.0f} ({charts.cancer_source_label(u)}, {d[:7]})" for v, u, d in zip(g.n, g.source_url, g.source_date))
    print(f"{k}:\n  {vals}\n")
charts.national_counts_cancer(nat).show()

Working radiotherapy machines (any type):
  10 (ARCON via Leadership, 2022-09), 20 (ARCON via The Guardian, 2022-09), 4 (Experts via The Guardian, 2023-02), 15 (IARC 2025 via BusinessDay, 2026-02), 10 (Nigerian Cancer Society, 2026-02), 10 (Daily Trust, 2026-04), 12 (Modern Ghana (unnamed stakeholders), 2026-11)

Working linacs:
  4 (JCO Glob Oncol 2021, 2021-08), 4 (SAGE Open Med 2023, 2023-02), 14 (JCO Glob Oncol 2024, 2024-01), 3 (Ann Med Surg 2025, 2025-10)

Oncologists:
  100 (ARCON via The Guardian, 2022-09), 28 (SAGE Open Med 2023, 2023-02), 72 (JCO Glob Oncol 2024, 2024-01), 100 (ARCON president, 2025-01), 44 (Ann Med Surg 2025, 2025-10), 70 (Nigerian Cancer Society, 2026-02)

Machines needed:
  157 (Lancet Oncol 2021 (IAEA DIRAC), 2021-09), 211 (Lancet Oncol 2021 (IAEA DIRAC), 2021-09), 120 (IARC 2025 via BusinessDay, 2026-02)



In [6]:
charts.register_gap_types(by_state, "Cancer").show()

## 3. One radiotherapy centre: does the code match a hand calculation?
`tests/test_centre_model.py` works one case on paper: 1 linac, 1 shift, 40 fractions a shift, 5 days, half
the slots used, 20 fractions a course, N1,000,000 a course, N1,000 to the dollar.

In [7]:
sys.path.insert(0, str(ROOT / "tests"))
from test_centre_model import radiotherapy_case
r = M.run(radiotherapy_case())
by_hand = {"weekly_capacity": 200, "sessions_per_year": 5_200, "patients_treated": 260,
           "variable_cost_per_session_ngn": 7_000, "fixed_cost_per_year_ngn": 126_000_000,
           "net_cash_per_year_ngn": 97_600_000, "breakeven_capacity_used": 0.28175,
           "breakeven_price_per_course_ngn": 624_615.38, "payback_years": 15.37, "cost_per_patient_year_ngn": 1_201_538.46}
check = pd.DataFrame({"by hand": by_hand, "code": {k: round(r[k], 5) for k in by_hand}})
check["match"] = [math.isclose(h, c, rel_tol=1e-3) for h, c in zip(check["by hand"], check["code"])]
print("All match:", check.match.all())
check

All match: True


,by hand,code,match
weekly_capacity,2.000000e+02,2.000000e+02,True
sessions_per_year,5.200000e+03,5.200000e+03,True
patients_treated,2.600000e+02,2.600000e+02,True
variable_cost_per_session_ngn,7.000000e+03,7.000000e+03,True
fixed_cost_per_year_ngn,1.260000e+08,1.260000e+08,True
net_cash_per_year_ngn,9.760000e+07,9.760000e+07,True
breakeven_capacity_used,2.817500e-01,2.817500e-01,True
breakeven_price_per_course_ngn,6.246154e+05,6.246154e+05,True
payback_years,1.537000e+01,1.536885e+01,True
cost_per_patient_year_ngn,1.201538e+06,1.201538e+06,True


## 4. The inputs, and where each comes from

In [8]:
a = M.load_assumptions(M.ASSUMPTIONS_CANCER)
rows = [{"input": n, "base": M._resolve(s["value"]), "low": M._resolve(s["low"]), "high": M._resolve(s["high"]),
         "unit": s["unit"], "sourced": not str(s["source"]).startswith("assumption")} for _, n, s in M.input_specs(a)]
inputs = pd.DataFrame(rows)
print(f"{inputs.sourced.sum()} of {len(inputs)} inputs have a public source (some only partly); "
      f"{(~inputs.sourced).sum()} are assumptions to verify.")
inputs

11 of 22 inputs have a public source (some only partly); 11 are assumptions to verify.


,input,base,low,high,unit,sourced
0,machines,1,1,2,linear accelerators (linacs),True
1,shifts_per_day,2,1,2,shifts per machine per day,True
2,sessions_per_machine_per_shift,40,30,50,fractions (treatments) per linac per 8-hour shift,True
3,days_open_per_week,5,5,6,days per week,False
4,capacity_used,0.75,0.5,0.9,share of available treatment slots used (0-1),False
5,sessions_per_course,20,15,25,fractions per course of external-beam radiotherapy,True
6,price_out_of_pocket_ngn,800000,650000,1200000,NGN per course,True
7,price_subsidised_ngn,800000,400000,800000,NGN per course received by the centre,False
8,price_nhia_ngn,800000,400000,800000,NGN per course received by the centre,True
9,price_employer_hmo_ngn,1200000,1000000,1800000,NGN per course,False


## 5. Results: low / base / high, and a weaker naira

In [9]:
sc = pd.read_csv(P / "centre_model_cancer_scenarios.csv")
charts.centre_scenarios(sc).show()
b = sc[(sc.scenario == "base") & (sc.naira_shock == 0)].iloc[0]
print(f"Base case: {b.patients_treated:,.0f} patients a year ({b.sessions_per_year:,.0f} fractions), "
      f"{b.capacity_used:.0%} of slots used.")
print(f"Price received N{b.average_price_per_course_ngn:,.0f} a course; break-even N{b.breakeven_price_per_course_ngn:,.0f}.")
v = b.net_cash_per_year_ngn
print(f"Net cash a year: {'−' if v < 0 else ''}N{abs(v) / 1e6:,.0f}m; set-up capital N{b.setup_capital_ngn / 1e9:,.2f}bn.")

Base case: 780 patients a year (15,600 fractions), 75% of slots used.
Price received N820,000 a course; break-even N907,833.
Net cash a year: −N69m; set-up capital N4.56bn.


In [10]:
sens = pd.read_csv(P / "centre_model_cancer_sensitivity.csv")
charts.centre_sensitivity(sens, source=charts.SRC_MODEL_CANCER).show()
for _, s in sens.head(5).iterrows():
    print(f"  {s.input:40} swing N{s.swing_ngn / 1e6:,.0f}m a year ({'sourced' if s.verified else 'ASSUMPTION'})")

  maintenance_pct_of_equipment_per_year    swing N395m a year (sourced)
  equipment_cost_per_machine_usd           swing N380m a year (sourced)
  price_out_of_pocket_ngn                  swing N365m a year (sourced)
  consumables_per_session_usd              swing N308m a year (ASSUMPTION)
  capacity_used                            swing N236m a year (ASSUMPTION)


## 6. Two illustrative PPP structures
> **Illustrative structures, general examples only. Not any real firm's terms.**

In [11]:
ppp = pd.read_csv(P / "centre_model_cancer_ppp.csv")
t = ppp.copy()
for c in [c for c in t.columns if c.endswith("_ngn")]:
    t[c] = t[c].map(lambda v: f"{'−' if v < 0 else ''}N{abs(v) / 1e6:,.1f}m")
t.drop(columns="label")

,naira_shock,structure,partner_income_ngn,partner_costs_ngn,partner_net_cash_ngn,partner_payback_years,hospital_income_ngn,hospital_costs_ngn,hospital_net_cash_ngn
0,0.0,fee,N468.0m,N422.1m,N45.9m,99.262002,N171.6m,N286.0m,−N114.4m
1,0.2,fee,N468.0m,N506.5m,−N38.5m,inf,N171.6m,N286.0m,−N114.4m
2,0.5,fee,N468.0m,N633.2m,−N165.2m,inf,N171.6m,N286.0m,−N114.4m
3,0.0,share,N447.7m,N422.1m,N25.6m,177.865119,N191.9m,N286.0m,−N94.1m
4,0.2,share,N447.7m,N506.5m,−N58.8m,inf,N191.9m,N286.0m,−N94.1m
5,0.5,share,N447.7m,N633.2m,−N185.4m,inf,N191.9m,N286.0m,−N94.1m


## 7. Dialysis and radiotherapy side by side (base case)

In [12]:
k = pd.read_csv(P / "centre_model_scenarios.csv").query("scenario == 'base' and naira_shock == 0").iloc[0]
side = pd.DataFrame({
    "dialysis centre (10 machines)": [k.patients_treated, k.capacity_used, k.net_cash_per_year_ngn / 1e6, k.setup_capital_ngn / 1e6],
    "radiotherapy centre (1 linac)": [b.patients_treated, b.capacity_used, b.net_cash_per_year_ngn / 1e6, b.setup_capital_ngn / 1e6]},
    index=["patients treated (dialysis: on treatment; radiotherapy: per year)", "capacity used",
           "net cash per year (N m)", "set-up capital (N m)"]).round(2)
side

,dialysis centre (10 machines),radiotherapy centre (1 linac)
patients treated (dialysis: on treatment; radiotherapy: per year),45.00,780.00
capacity used,0.75,0.75
net cash per year (N m),-92.44,-68.51
set-up capital (N m),607.35,4555.14


## 8. Checks

In [13]:
from load_register import check_register_rows
errors = (check_register_rows(pd.read_csv(ROOT / "data/register/specialist_capacity.csv", dtype=str, keep_default_na=False))
          + validate.check_assumptions(ROOT / "config/assumptions_cancer.yaml")
          + validate.check_payer_mix(ROOT / "config/assumptions_cancer.yaml"))
log = pd.read_csv(ROOT / "data/register/verification_log.csv")
print("Register rows all sourced and dated; cancer assumptions complete; payer mix sums to 1." if not errors else errors)
print(f"Spot-checked quotes: {len(log)} ({log.verdict.value_counts().to_dict()})")

Register rows all sourced and dated; cancer assumptions complete; payer mix sums to 1.
Spot-checked quotes: 20 ({'verified (exact)': 16, 'verified (near-exact)': 3, 'verified': 1})
